# AR-PCNN: Autoregressive Predictive + Physics-Constrained Neural Network
## Author: RIDWAN ADEMOLA, IBRAHIM

## IDA-KI OpenLab Research Bridge, TU Dresden — Middle T-Girder

### Overview

AR-PCNN combines a data-driven autoregressive prediction model with a physics-constrained reconstruction model to enable next-day full-field structural response prediction.

---

### Architecture

#### Step 1: AR(1) Cross-Sensor Predictive Model

Predicts next-day Tilt-X at both sensor locations using:

- Previous-day Tilt-X from both sensors
- Temperature
- Temperature change (dT)
- Front–back temperature difference (T_diff)
- Loading phase

Training period:

- 22 Feb – 21 Sep 2024
- 213 daily observations

Prediction period:

- 22 Sep – 31 Oct 2024
- 40 daily observations

---

#### Step 2: CEI-PCNN Reconstructive Model

Uses the same EnvelopeNet + BeamNet architecture as the CEI-PCNN model.

Key characteristics:

- Trained using measured Tilt-X only
- Training period: 213 days
- Learns the mapping from sensor measurements to full-field structural response

During inference, the model accepts:

- Measured Tilt-X values
- AR-predicted Tilt-X values

---

#### Step 3: End-to-End Pipeline

**Training Period**

```text
Measured Tilt-X
        ↓
      PCNN
        ↓
Full-Field Reconstruction
```

**Test Period**

```text
AR-Predicted Tilt-X
          ↓
        PCNN
          ↓
Full-Field Prediction
```

Predictions are evaluated against FEM reconstructions at:

- x = 5 m
- x = 10 m
- x = 20 m
- x = 25 m

---

### Deployment Workflow

```text
Past 7 Days Measured Tilt-X
            +
 Forecast Temperature
            +
      Known Phase
            ↓
        AR Model
            ↓
  Predicted Tilt-X
            ↓
      Trained PCNN
            ↓
 Full-Field Prediction
```

---

### Output

The AR model provides next-day sensor predictions, while the PCNN converts those predictions into full-field slope and deflection estimates across the bridge.

This notebook contains the complete executable implementation of the AR-PCNN framework, including data processing, AR model training, PCNN training, inference, evaluation, and visualization. Run all cells sequentially to reproduce the reported results.

## Imports, Constants, and Configuration

Core libraries, physical constants (EI, L), Fourier frequencies, validation positions, colour schemes, and output directory.

In [1]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.linear_model import Ridge
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, json
from datetime import datetime, timedelta

torch.set_default_dtype(torch.float64)

data_wd = "C:\\Users\\ridoc\\OneDrive\\Desktop\\Folders\\SMACCs\\Thesis\\Physics Guided Framework\\Draft\\cleaned_tilt_data.csv"
FEA_SLOPE_CSV = 'outputs/processing/Tiltx/FEM/reconstructed_slopes.csv'
FEA_DEFL_CSV  = 'outputs/processing/Tiltx/FEM/reconstructed_deflections.csv'
OUT = os.path.join(os.path.dirname(data_wd), "Models","outputs","processing","Tiltx","AR_PCNN")
os.makedirs(OUT, exist_ok=True)

L_TOTAL = 30.0; L_SPAN = 15.0
EI_NOM = 37.0e6 * 5.20833e-3; EI_INV = 1.0 / EI_NOM
TILTX_SCALE = 1000.0
FOURIER_FREQS = [1, 2, 4, 8, 16]; N_FOURIER = len(FOURIER_FREQS)
FOURIER_FREQS_X = [1, 2, 3]  # LOW frequencies for envelope — avoids derivative oscillation

VAL_POS = [5.0, 10.0, 20.0, 25.0]
CP = {1: '#2196F3', 2: '#4CAF50', 3: '#FF9800', 4: '#F44336'}
LP = {1: 'Ph1', 2: 'Ph2: +tracks', 3: 'Ph3: +veh4.1t', 4: 'Ph4: +veh9t'}
DC = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
      '#42d4f4', '#f032e6', '#bfef45', '#800000', '#469990']
REF_DATE = pd.Timestamp('2024-02-01').date()

## Fourier Temporal Encoding

Encodes scalar values (normalised time t, standardised temperature T) as sin/cos pairs at frequencies [1,2,4,8,16]. This overcomes the spectral bias of standard MLPs and injects prior knowledge that structural thermal response is smooth and quasi-periodic. Each scalar becomes a 10-dimensional vector (2 features x 5 frequencies).

In [2]:
def fourier_encode(vals, freqs):
    p = []
    for f in freqs:
        p.append(torch.sin(2*np.pi*f*vals))
        p.append(torch.cos(2*np.pi*f*vals))
    return torch.cat(p, dim=1)

## Fourier Spatial Encoding (for EnvelopeNet)

Encodes the spatial coordinate x using LOW frequencies [1,2,3] to avoid derivative oscillation through the 4th-order PDE. Higher Fourier frequencies would produce high-frequency oscillations in the 4th derivative of the envelope, destabilising the PDE loss. Includes normalised x/L as an additional feature (7 total).

In [3]:
def fourier_x(x):
    """Fourier features for spatial coordinate (for EnvelopeNet input).
    Uses low frequencies [1,2,3] to avoid derivative oscillation."""
    p = [x / L_TOTAL]  # normalised x
    for f in FOURIER_FREQS_X:
        p.append(torch.sin(2*np.pi*f*x/L_TOTAL))
        p.append(torch.cos(2*np.pi*f*x/L_TOTAL))
    return torch.cat(p, dim=1)  # 1 + 2*3 = 7 features

## EnvelopeNet Architecture

3-hidden-layer, 64-neuron tanh network that learns both span deflection shapes as a single smooth function. Input: 7 Fourier spatial features. Output: scalar envelope value. Frozen after pretraining — serves as the trial function during main training.

The learned function approximates:
- Span AB: x^2(L-x)^2 (fixed-fixed deflection shape)
- Span BC: xi^2(L-xi)(1.5L-xi) (fixed-pinned deflection shape)

In [4]:
class EnvelopeNet(nn.Module):
    """Pretrained NN that approximates the beam deflection shape.
    Learns the analytical shapes for both spans as a single smooth function:
      Span AB: x^2*(L-x)^2  (fixed-fixed)
      Span BC: xi^2*(L-xi)*(1.5L-xi)  (fixed-pinned)
    Uses LOW Fourier frequencies for x to avoid derivative oscillation.
    Deeper network (3 hidden layers) compensates for fewer Fourier features.
    FROZEN during main training — only provides the spatial template."""
    def __init__(self):
        super().__init__()
        n_fx = 1 + 2 * len(FOURIER_FREQS_X)  # 7
        self.net = nn.Sequential(
            nn.Linear(n_fx, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(fourier_x(x))

## EnvelopeNet Pretraining (10,000 epochs)

Trains the envelope to match analytical beam shapes with 4 loss components:
1. **L_shape:** MSE against piecewise target (normalised to [0,1])
2. **L_BC:** env(0)=env(15)=env(30)=0, env'(0)=env'(15)=0
3. **L_mom:** env''(30)=0 (partially hardcodes the moment BC)
4. **L_d4:** Variance of env'''' within each span (constancy penalty, ramped in after epoch 2000)

The d4 constancy penalty is critical: for polynomial deflection shapes, the 4th derivative is a constant. If the envelope deviates from this, the PDE loss will see spurious spatial variation in the residual. After training, all parameters are frozen (`requires_grad = False`).

In [5]:
def pretrain_envelope(n_epochs=15000, lr=3e-3):
    """Pretrain EnvelopeNet to match analytical beam deflection shapes.
    Uses low Fourier frequencies and derivative smoothness penalty."""
    print("  Pretraining EnvelopeNet...")
    env = EnvelopeNet()
    opt = torch.optim.Adam(env.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-5)

    # Target: piecewise analytical shapes
    x_train = np.linspace(0, L_TOTAL, 2000)
    y_target = np.zeros_like(x_train)
    for i, x in enumerate(x_train):
        if x <= L_SPAN:
            y_target[i] = x**2 * (L_SPAN - x)**2
        else:
            xi = x - L_SPAN
            y_target[i] = xi**2 * (L_SPAN - xi) * (1.5*L_SPAN - xi)
    y_target /= np.max(np.abs(y_target))

    xt = torch.tensor(x_train.reshape(-1, 1))
    yt = torch.tensor(y_target.reshape(-1, 1))

    for ep in range(n_epochs):
        pred = env(xt); loss = torch.mean((pred - yt)**2)
        # BC enforcement
        xbc = torch.tensor([[0.0], [L_SPAN], [L_TOTAL]]).requires_grad_(True)
        pbc = env(xbc); loss_bc = torch.mean(pbc**2) * 500
        dpbc = torch.autograd.grad(pbc.sum(), xbc, create_graph=True)[0]
        loss_d = (dpbc[0]**2 + dpbc[1]**2).squeeze() * 500
        # Enforce env''(30) = 0 (zero moment at pinned support)
        # This makes w''(30) partially hard: eliminates env''*NN term,
        # leaving only 2*env'(30)*NN'(30) for the soft loss to handle.
        x30_bc = torch.tensor([[L_TOTAL]]).requires_grad_(True)
        e30 = env(x30_bc)
        de30_1 = torch.autograd.grad(e30, x30_bc, create_graph=True)[0]
        de30_2 = torch.autograd.grad(de30_1, x30_bc, create_graph=True)[0]
        loss_d = loss_d + de30_2.squeeze()**2 * 500

        # d4 CONSTANCY penalty: env'''' should be constant per span
        # (for x^2*(L-x)^2 and xi^2*(L-xi)*(1.5L-xi), d4 = 24/scale = const)
        # Penalise variance of d4 across collocation points
        if ep >= 2000:  # start after shape is roughly learned
            n_d4 = 15
            xd4_AB = torch.linspace(1.0, 14.0, n_d4).view(-1, 1).requires_grad_(True)
            ed4_AB = env(xd4_AB); o4 = torch.ones(n_d4, 1)
            de1 = torch.autograd.grad(ed4_AB, xd4_AB, o4, create_graph=True)[0]
            de2 = torch.autograd.grad(de1, xd4_AB, o4, create_graph=True)[0]
            de3 = torch.autograd.grad(de2, xd4_AB, o4, create_graph=True)[0]
            de4 = torch.autograd.grad(de3, xd4_AB, o4, create_graph=True)[0]
            xd4_BC = torch.linspace(16.0, 29.0, n_d4).view(-1, 1).requires_grad_(True)
            ed4_BC = env(xd4_BC)
            de1b = torch.autograd.grad(ed4_BC, xd4_BC, o4, create_graph=True)[0]
            de2b = torch.autograd.grad(de1b, xd4_BC, o4, create_graph=True)[0]
            de3b = torch.autograd.grad(de2b, xd4_BC, o4, create_graph=True)[0]
            de4b = torch.autograd.grad(de3b, xd4_BC, o4, create_graph=True)[0]
            # Penalise variance (d4 should be constant within each span)
            loss_d4 = torch.var(de4.squeeze()) + torch.var(de4b.squeeze())
            lam_d4 = min((ep - 2000) * 50.0 / n_epochs, 50.0)  # ramp up
        else:
            loss_d4 = torch.tensor(0.0); lam_d4 = 0.0

        total = loss + loss_bc + loss_d + lam_d4 * loss_d4
        opt.zero_grad(); total.backward(); opt.step(); sched.step()
        if ep % 3000 == 0:
            print(f"    ep {ep}: shape={loss.item():.7f} d4var={loss_d4.item():.6f}")

    # Freeze all parameters
    for p in env.parameters():
        p.requires_grad_(False)

    # Verify
    print(f"    Final shape_loss={loss.item():.6f}")
    for xv in [0, 7.5, 15, 22.5, 30]:
        with torch.no_grad():
            v = env(torch.tensor([[float(xv)]])).item()
        idx = np.argmin(np.abs(x_train - xv))
        print(f"    env({xv:.1f})={v:+.5f} target={y_target[idx]:+.5f}")
    return env

## Main Neural Network (BeamNet)

4-hidden-layer, 64-neuron tanh network.

**Input (26 features):** x(1) + env(x)(1) + Fourier_t(10) + Fourier_T(10) + phase_oh(4)

**Initialisation:** Xavier normal with gain=0.3 — small initial weights keep NN output near zero at epoch 0, so w = phi * NN ~ 0 (physically reasonable initial state).

**Why tanh?** The PDE requires computing w'''' (4th derivative) through the network. Tanh is C-infinity, so all derivatives exist and are smooth. ReLU would give zero 2nd derivatives.

In [6]:
class BeamNet(nn.Module):
    """Main trainable NN. Input: (x, env(x), fourier(t), fourier(T), phase_oh).
    env(x) as extra input gives the NN distance-to-boundary information."""
    def __init__(self):
        super().__init__()
        # x(1) + env(1) + fourier_t(10) + fourier_T(10) + phase(4) = 26
        n_in = 1 + 1 + 2*N_FOURIER + 2*N_FOURIER + 4
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=0.3); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

## CEI-PCNN Solver Class (EnvelopeNet Architecture)

Identical architecture to standalone PETF. The solver:
1. Pretrains EnvelopeNet (10k epochs) to learn the beam deflection shape
2. Freezes the envelope
3. Trains BeamNet + load parameters with 4-component loss (data, PDE, continuity, moment)
4. Uses Adam (5000 epochs) + L-BFGS (80 steps)

The key difference from standalone PETF: the sensor input tiltx includes AR-predicted values for the test period (days 214-253), not measured values. The PCNN treats all 253 days identically — it cannot distinguish measured from predicted input.

In [7]:
class CEI_PCNN:
    def __init__(self, N, phases, T_arr):
        self.N = N
        self.xi_AB = torch.linspace(0.5, 14.5, 20)
        self.xi_BC = torch.linspace(15.5, 29.5, 20)
        self.t_norm = torch.arange(N, dtype=torch.float64) / (N - 1)
        T = np.array(T_arr); self.T_norm = torch.tensor((T - T.mean()) / (T.std() + 1e-10))
        pa = np.array(phases); ph_oh = torch.zeros(N, 4)
        for i in range(N): ph_oh[i, pa[i]-1] = 1.0
        self.ph_oh = ph_oh
        w = np.zeros(N)
        for ph in [1,2,3,4]: mask = pa == ph; w[mask] = 1.0 / mask.sum()
        w /= w.sum(); self.sw = torch.tensor(w)
        self._net = None; self._env = None

    def _inp(self, x, idx):
        """Build input: (x, env(x), fourier_t, fourier_T, phase_oh)"""
        ft = fourier_encode(self.t_norm[idx].view(-1,1), FOURIER_FREQS)
        fT = fourier_encode(self.T_norm[idx].view(-1,1), FOURIER_FREQS)
        with torch.no_grad(): env_val = self._env(x)
        return torch.cat([x, env_val, ft, fT, self.ph_oh[idx]], dim=1)

    def _eval_w(self, x, idx, net):
        """w = EnvelopeNet(x) * MainNN(x, env(x), t, T, phase)"""
        env_val = self._env(x)  # with grad for derivatives
        inp = self._inp(x, idx)
        inp_d = torch.cat([x, env_val, inp[:, 2:]], dim=1)  # reconnect differentiable env
        return env_val * net(inp_d)

    def solve(self, m11, m19, n_adam=5000, n_lbfgs=80, bs=32):
        # Pretrain envelope
        env = pretrain_envelope(n_epochs=10000)
        self._env = env
        
        net = BeamNet()
        q_AB = nn.Parameter(torch.full((self.N,), 5.0))
        q_BC = nn.Parameter(torch.full((self.N,), 5.0))
        s11 = torch.tensor(m11); s19 = torch.tensor(m19)
        opt = torch.optim.Adam([{'params': net.parameters(), 'lr': 1e-3},
                                 {'params': [q_AB, q_BC], 'lr': 3e-1}])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_adam, eta_min=1e-6)
        hist = {'data': [], 'pde': [], 'cont': [], 'mom': [], 'total': []}
        nc_per = 8

        for ep in range(n_adam):
            idx = torch.multinomial(self.sw, bs, replacement=True); opt.zero_grad()
            # Data
            x1 = torch.full((bs,1), 11.0).requires_grad_(True)
            w1 = self._eval_w(x1, idx, net)
            dw1 = torch.autograd.grad(w1.sum(), x1, create_graph=True)[0]
            x2 = torch.full((bs,1), 19.0).requires_grad_(True)
            w2 = self._eval_w(x2, idx, net)
            dw2 = torch.autograd.grad(w2.sum(), x2, create_graph=True)[0]
            Ld = torch.mean((-dw1.squeeze()*1000 - s11[idx])**2) + \
                 torch.mean((-dw2.squeeze()*1000 - s19[idx])**2)
            # PDE
            idx_c = idx.repeat_interleave(nc_per); o = torch.ones(bs*nc_per, 1)
            ci = torch.randint(0, 20, (bs*nc_per,))
            xc = self.xi_AB[ci].view(-1,1).requires_grad_(True)
            wc = self._eval_w(xc, idx_c, net)
            d1 = torch.autograd.grad(wc, xc, o, create_graph=True)[0]
            d2 = torch.autograd.grad(d1, xc, o, create_graph=True)[0]
            d3 = torch.autograd.grad(d2, xc, o, create_graph=True)[0]
            d4 = torch.autograd.grad(d3, xc, o, create_graph=True)[0]
            Lpa = torch.mean((d4 + q_AB[idx].repeat_interleave(nc_per).view(-1,1)*EI_INV).squeeze()**2)
            ci2 = torch.randint(0, 20, (bs*nc_per,))
            xc2 = self.xi_BC[ci2].view(-1,1).requires_grad_(True)
            wc2 = self._eval_w(xc2, idx_c, net)
            d1b = torch.autograd.grad(wc2, xc2, o, create_graph=True)[0]
            d2b = torch.autograd.grad(d1b, xc2, o, create_graph=True)[0]
            d3b = torch.autograd.grad(d2b, xc2, o, create_graph=True)[0]
            d4b = torch.autograd.grad(d3b, xc2, o, create_graph=True)[0]
            Lpb = torch.mean((d4b + q_BC[idx].repeat_interleave(nc_per).view(-1,1)*EI_INV).squeeze()**2)
            Lp = Lpa + Lpb
            # Continuity
            o1 = torch.ones(bs, 1)
            xL = torch.full((bs,1), 14.7).requires_grad_(True)
            wL = self._eval_w(xL, idx, net)
            d1L = torch.autograd.grad(wL, xL, o1, create_graph=True)[0]
            d2L = torch.autograd.grad(d1L, xL, o1, create_graph=True)[0]
            d3L = torch.autograd.grad(d2L, xL, o1, create_graph=True)[0]
            xR = torch.full((bs,1), 15.3).requires_grad_(True)
            wR = self._eval_w(xR, idx, net)
            d1R = torch.autograd.grad(wR, xR, o1, create_graph=True)[0]
            d2R = torch.autograd.grad(d1R, xR, o1, create_graph=True)[0]
            d3R = torch.autograd.grad(d2R, xR, o1, create_graph=True)[0]
            Lc = torch.mean((d2L.squeeze()-d2R.squeeze())**2) + torch.mean((d3L.squeeze()-d3R.squeeze())**2)
            # Moment BC
            x30 = torch.full((bs,1), 29.9).requires_grad_(True)
            w30 = self._eval_w(x30, idx, net)
            d1_30 = torch.autograd.grad(w30, x30, o1, create_graph=True)[0]
            d2_30 = torch.autograd.grad(d1_30, x30, o1, create_graph=True)[0]
            Lm = torch.mean(d2_30.squeeze()**2)

            lp = min(0.1 + ep / n_adam, 1.0); lc = min(1 + ep*20/n_adam, 20.0)
            loss = 20*Ld + lp*Lp + lc*Lc + lc*Lm
            loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step(); sched.step()
            hist['data'].append(Ld.item()); hist['pde'].append(Lp.item())
            hist['cont'].append(Lc.item()); hist['mom'].append(Lm.item())
            hist['total'].append(loss.item())
            if ep % 1000 == 0:
                print(f"    ep {ep}: Ld={Ld.item():.3e} Lp={Lp.item():.3e} Lc={Lc.item():.3e} Lm={Lm.item():.3e}")

        # L-BFGS
        print(f"    L-BFGS ({n_lbfgs} steps)...")
        lbfgs = torch.optim.LBFGS(list(net.parameters())+[q_AB,q_BC], lr=0.3, max_iter=20,
                                    line_search_fn='strong_wolfe')
        _self = self  # capture for closure
        for step in range(n_lbfgs):
            def closure():
                lbfgs.zero_grad(); bs2 = min(64, _self.N)
                idx2 = torch.multinomial(_self.sw, bs2, replacement=True)
                x1=torch.full((bs2,1),11.0).requires_grad_(True)
                w1=_self._eval_w(x1,idx2,net); dw1=torch.autograd.grad(w1.sum(),x1,create_graph=True)[0]
                x2=torch.full((bs2,1),19.0).requires_grad_(True)
                w2=_self._eval_w(x2,idx2,net); dw2=torch.autograd.grad(w2.sum(),x2,create_graph=True)[0]
                Ld=torch.mean((-dw1.squeeze()*1000-s11[idx2])**2)+torch.mean((-dw2.squeeze()*1000-s19[idx2])**2)
                idx_c2=idx2.repeat_interleave(8); o=torch.ones(bs2*8,1)
                xca=_self.xi_AB[torch.randint(0,20,(bs2*8,))].view(-1,1).requires_grad_(True)
                wca=_self._eval_w(xca,idx_c2,net)
                d1=torch.autograd.grad(wca,xca,o,create_graph=True)[0]; d2=torch.autograd.grad(d1,xca,o,create_graph=True)[0]
                d3=torch.autograd.grad(d2,xca,o,create_graph=True)[0]; d4=torch.autograd.grad(d3,xca,o,create_graph=True)[0]
                Lpa=torch.mean((d4+q_AB[idx2].repeat_interleave(8).view(-1,1)*EI_INV).squeeze()**2)
                xcb=_self.xi_BC[torch.randint(0,20,(bs2*8,))].view(-1,1).requires_grad_(True)
                wcb=_self._eval_w(xcb,idx_c2,net)
                d1b=torch.autograd.grad(wcb,xcb,o,create_graph=True)[0]; d2b=torch.autograd.grad(d1b,xcb,o,create_graph=True)[0]
                d3b=torch.autograd.grad(d2b,xcb,o,create_graph=True)[0]; d4b=torch.autograd.grad(d3b,xcb,o,create_graph=True)[0]
                Lpb=torch.mean((d4b+q_BC[idx2].repeat_interleave(8).view(-1,1)*EI_INV).squeeze()**2)
                o1=torch.ones(bs2,1)
                xL=torch.full((bs2,1),14.7).requires_grad_(True)
                wL=_self._eval_w(xL,idx2,net)
                d1L=torch.autograd.grad(wL,xL,o1,create_graph=True)[0]; d2L=torch.autograd.grad(d1L,xL,o1,create_graph=True)[0]
                d3L=torch.autograd.grad(d2L,xL,o1,create_graph=True)[0]
                xR=torch.full((bs2,1),15.3).requires_grad_(True)
                wR=_self._eval_w(xR,idx2,net)
                d1R=torch.autograd.grad(wR,xR,o1,create_graph=True)[0]; d2R=torch.autograd.grad(d1R,xR,o1,create_graph=True)[0]
                d3R=torch.autograd.grad(d2R,xR,o1,create_graph=True)[0]
                Lc=torch.mean((d2L.squeeze()-d2R.squeeze())**2)+torch.mean((d3L.squeeze()-d3R.squeeze())**2)
                x30=torch.full((bs2,1),29.9).requires_grad_(True)
                w30=_self._eval_w(x30,idx2,net)
                d1_30=torch.autograd.grad(w30,x30,o1,create_graph=True)[0]
                d2_30=torch.autograd.grad(d1_30,x30,o1,create_graph=True)[0]
                Lm=torch.mean(d2_30.squeeze()**2)
                loss=20*Ld+Lpa+Lpb+20*Lc+20*Lm; loss.backward(); return loss
            try: lbfgs.step(closure)
            except: break
            if step % 20 == 0: print(f"    L-BFGS {step}")

        self._net = net; self._q_AB = q_AB; self._q_BC = q_BC
        return hist

    def predict_at(self, positions):
        net = self._net; N = self.N; P = len(positions)
        slopes = np.zeros((N, P)); defls = np.zeros((N, P))
        idx_all = torch.arange(N)
        for p, xv in enumerate(positions):
            x_t = torch.full((N,1), float(xv)).requires_grad_(True)
            w = self._eval_w(x_t, idx_all, net)
            dw = torch.autograd.grad(w.sum(), x_t)[0]
            slopes[:, p] = (-dw.squeeze()*TILTX_SCALE).detach().numpy()
            defls[:, p] = (-w.squeeze()*1000).detach().numpy()
        return slopes, defls

## Accuracy Metrics

Computes R2, RMSE, MAE, MAPE, Bias, and Pearson r. MAPE is skipped when values are near zero. Pearson r captures shape agreement independent of offset/scale.

In [8]:
def met(y, yp):
    err = yp - y; ss = np.sum(err**2); st = np.sum((y-y.mean())**2)
    mask = np.abs(y) > 1e-6
    mape = float(np.mean(np.abs(err[mask]/y[mask]))*100) if mask.sum() > 0 else float('nan')
    return {'R2': float(1-ss/(st+1e-12)), 'RMSE': float(np.sqrt(np.mean(err**2))),
            'MAE': float(np.mean(np.abs(err))), 'MAPE': mape,
            'Bias': float(np.mean(err)),
            'r': float(np.corrcoef(y.ravel(), yp.ravel())[0,1]) if y.ravel().std()>1e-10 else 0.0}

## Physics Compliance Evaluation

Evaluates three categories after training:
1. **BC violations** (RMS over all N days): Hard BCs should be ~1e-10, soft w''(30) depends on loss quality
2. **Continuity** at interior support: moment and shear jumps should be zero
3. **PDE residual**: w'''' + q/EI at collocation points — small residual means the solution is genuinely a beam deflection

RMS is used (not mean) because it penalises large individual violations more heavily.

In [9]:
def evaluate_physics(solver, N):
    net = solver._net; env = solver._env; idx_all = torch.arange(N); results = {}
    bc_v = {}
    for xv, bc in [(0.0,'w(0)'), (15.0,'w(15)'), (30.0,'w(30)')]:
        x_t = torch.full((N,1), xv).requires_grad_(True)
        w = solver._eval_w(x_t, idx_all, net)
        bc_v[bc] = float(torch.sqrt(torch.mean(w**2)).item())
    for xv, bc in [(0.0,"w'(0)"), (15.0,"w'(15)")]:
        x_t = torch.full((N,1), xv).requires_grad_(True)
        w = solver._eval_w(x_t, idx_all, net)
        dw = torch.autograd.grad(w.sum(), x_t)[0]
        bc_v[bc] = float(torch.sqrt(torch.mean(dw**2)).item())
    x30 = torch.full((N,1), 29.95).requires_grad_(True)
    w30 = solver._eval_w(x30, idx_all, net)
    d1 = torch.autograd.grad(w30.sum(), x30, create_graph=True)[0]
    d2 = torch.autograd.grad(d1.sum(), x30)[0]
    bc_v["w''(30)"] = float(torch.sqrt(torch.mean(d2**2)).item())
    results['bc_violations_rms'] = bc_v

    o1 = torch.ones(N, 1)
    xL = torch.full((N,1), 14.7).requires_grad_(True)
    wL = solver._eval_w(xL, idx_all, net)
    d1L = torch.autograd.grad(wL, xL, o1, create_graph=True)[0]
    d2L = torch.autograd.grad(d1L, xL, o1, create_graph=True)[0]
    d3L = torch.autograd.grad(d2L, xL, o1)[0]
    xR = torch.full((N,1), 15.3).requires_grad_(True)
    wR = solver._eval_w(xR, idx_all, net)
    d1R = torch.autograd.grad(wR, xR, o1, create_graph=True)[0]
    d2R = torch.autograd.grad(d1R, xR, o1, create_graph=True)[0]
    d3R = torch.autograd.grad(d2R, xR, o1)[0]
    mj = d2L.squeeze().detach().numpy() - d2R.squeeze().detach().numpy()
    sj = d3L.squeeze().detach().numpy() - d3R.squeeze().detach().numpy()
    results['continuity'] = {'moment_rms': float(np.sqrt(np.mean(mj**2))), 'moment_max': float(np.max(np.abs(mj))),
                              'shear_rms': float(np.sqrt(np.mean(sj**2))), 'shear_max': float(np.max(np.abs(sj)))}

    pde = {}
    for sn, xi_ev, qp in [('AB', torch.linspace(1,14,20), solver._q_AB),
                            ('BC', torch.linspace(16,29,20), solver._q_BC)]:
        res_all = []
        for i in range(N):
            x_t = xi_ev.view(-1,1).clone().requires_grad_(True)
            idx_i = torch.full((20,), i, dtype=torch.long)
            ww = solver._eval_w(x_t, idx_i, net)
            oo = torch.ones_like(ww)
            d1 = torch.autograd.grad(ww, x_t, oo, create_graph=True)[0]
            d2 = torch.autograd.grad(d1, x_t, oo, create_graph=True)[0]
            d3 = torch.autograd.grad(d2, x_t, oo, create_graph=True)[0]
            d4 = torch.autograd.grad(d3, x_t, oo)[0]
            r = d4.squeeze().detach().numpy() + qp[i].detach().item()*EI_INV
            res_all.append(np.mean(r**2))
        pde[sn] = {'rms': float(np.sqrt(np.mean(res_all))), 'max': float(np.sqrt(np.max(res_all)))}
    results['pde_residual'] = pde
    return results

## Helper Functions

Date formatting and phase-transition vertical line markers for plots.

In [10]:
def day_to_date(d): return (pd.Timestamp('2024-02-01') + timedelta(days=int(d))).strftime('%B %d')
def _vlines(ax):
    for d in [21,46,81,148]: ax.axvline(d, color='gray', ls='--', lw=0.8, alpha=0.5)

## Data Loading

Loads cleaned tilt data CSV, computes daily averages, assigns loading phases, and merges with FEM baseline slopes/deflections at all spatial positions (0.5m intervals) for validation. Temperature T = mean of front/back thermocouples at x=11m.

In [11]:
def load_data():
    raw = pd.read_csv(data_wd); raw['Timestamp'] = pd.to_datetime(raw['Timestamp'])
    raw['T'] = (raw['temp_at_11m_front']+raw['temp_at_11m_back'])/2
    raw['day'] = (raw['Timestamp'].dt.date - REF_DATE).apply(lambda x: x.days)
    daily = raw.groupby('day').agg({'tiltx_11m':'mean','tiltx_19m':'mean','T':'mean'}).reset_index()
    daily = daily[daily['day']>=21].copy().reset_index(drop=True)
    daily['phase']=1; daily.loc[daily['day']>=46,'phase']=2
    daily.loc[daily['day']>=81,'phase']=3; daily.loc[daily['day']>=148,'phase']=4
    fea_sl = pd.read_csv(FEA_SLOPE_CSV); slope_cols = [c for c in fea_sl.columns if c.startswith('slope_')]
    daily = daily.merge(fea_sl[['day','net_udl_AB','net_udl_BC']+slope_cols], on='day', how='left')
    fea_df = pd.read_csv(FEA_DEFL_CSV); defl_cols = [c for c in fea_df.columns if c.startswith('defl_')]
    daily = daily.merge(fea_df[['day']+defl_cols], on='day', how='left')
    # Extra columns for AR model
    daily['dT_11'] = raw.groupby('day')['change_in_temp_11m'].mean().reindex(daily['day']).values
    daily['dT_19'] = raw.groupby('day')['change_in_temp_19m'].mean().reindex(daily['day']).values
    daily['T_diff'] = daily['T'].diff().fillna(0)
    for ph in [1,2,3,4]: print(f"  Phase {ph}: {(daily['phase']==ph).sum()} days")
    return daily, slope_cols, defl_cols

## Run: Load Data and Train

Load all data, instantiate the CEI-PCNN solver, and run the full training pipeline (Adam 5000 epochs + L-BFGS 80 steps). Expected runtime: 3-5 minutes on CPU.

In [12]:
t0 = datetime.now()
print("="*70)
print("AR-PCNN: Autoregressive Prediction + Physics-Constrained Reconstruction")
print("="*70)

daily, slope_cols, defl_cols = load_data()
N = len(daily); days = daily['day'].values; phases = daily['phase'].values
m11 = daily['tiltx_11m'].values; m19 = daily['tiltx_19m'].values; T = daily['T'].values
gpos = np.array([float(c.replace('slope_','').replace('m','')) for c in slope_cols])
fe_sl = daily[slope_cols].values; fe_df = daily[defl_cols].values
fe_q_AB = daily['net_udl_AB'].values; fe_q_BC = daily['net_udl_BC'].values

N_TEST = 40; N_TRAIN = N - N_TEST

# ═══════════════════════════════════════════════════════════════
# STEP 1: AR(1) CROSS-SENSOR PREDICTIVE MODEL
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"STEP 1: AR(1) Cross-Sensor Prediction")
print(f"  Train: {N_TRAIN} days (measured) | Test: {N_TEST} days (predicted)")
print(f"{'='*70}")

def build_ar_feat(df, s, e):
    T = df['T'].values[s:e]
    ph = df['phase'].values[s:e]
    return np.column_stack([
        df['tiltx_11m'].values[s-1:e-1], df['tiltx_19m'].values[s-1:e-1],
        T, df['T_diff'].values[s:e],
        df['dT_11'].values[s:e], df['dT_19'].values[s:e],
        (ph==2).astype(float), (ph==3).astype(float), (ph==4).astype(float),
        T*(ph==4).astype(float)])

BIAS_WINDOW = 14
Xtr_ar = build_ar_feat(daily, 1, N_TRAIN)
Xte_ar = build_ar_feat(daily, N_TRAIN, N)
Xl_ar = build_ar_feat(daily, N_TRAIN - BIAS_WINDOW, N_TRAIN)
ar_pred = {}; ar_metrics = {}
for col in ['tiltx_11m', 'tiltx_19m']:
    mdl = Ridge(alpha=0.1).fit(Xtr_ar, daily[col].values[1:N_TRAIN])
    pred_raw = mdl.predict(Xte_ar)
    # Bias correction
    y_last = daily[col].values[N_TRAIN - BIAS_WINDOW:N_TRAIN]
    bias_est = float(np.mean(mdl.predict(Xl_ar) - y_last))
    p = pred_raw - bias_est
    actual = daily[col].values[N_TRAIN:]
    rmse = np.sqrt(np.mean((p - actual)**2))
    r = np.corrcoef(actual, p)[0, 1]
    R2 = 1 - np.sum((p-actual)**2)/np.sum((actual-actual.mean())**2)
    ar_pred[col] = p
    ar_metrics[col] = {'rmse': rmse, 'r': r, 'R2': R2, 'model': mdl, 'bias_correction': bias_est}
    label = '11m' if '11' in col else '19m'
    print(f"  {label}: RMSE={rmse:.4f} R2={R2:.4f} r={r:.4f} (bias correction: {bias_est:+.4f})")
    coefs = mdl.coef_
    print(f"    AR_self={coefs[0]:.4f} AR_cross={coefs[1]:.4f} T={coefs[2]:.5f} ph4xT={coefs[9]:.5f}")

# Combined tiltx: measured (train) + predicted (test)
tiltx_11 = np.concatenate([m11[:N_TRAIN], ar_pred['tiltx_11m']])
tiltx_19 = np.concatenate([m19[:N_TRAIN], ar_pred['tiltx_19m']])

# ═══════════════════════════════════════════════════════════════
# STEP 2: CEI-PCNN (TRAINED ON MEASURED PERIOD ONLY)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"STEP 2: CEI-PCNN (trained on all {N} days: measured + AR-predicted)")
print(f"  Days 1-{N_TRAIN}: measured tiltx | Days {N_TRAIN+1}-{N}: AR-predicted tiltx")
print(f"{'='*70}")

# The PCNN sees ALL N observations for temporal features (t_norm, T_norm, phase)
# but trains its data loss on MEASURED tiltx (first N_TRAIN days only)
solver = CEI_PCNN(N, phases, T)
# The PCNN trains on ALL 253 days using:
#   - Measured tiltx for days 1-213 (training period)
#   - AR-predicted tiltx for days 214-253 (test period)
# The PCNN doesn't distinguish between measured and predicted input.
# This avoids temporal extrapolation in the Fourier features.
hist = solver.solve(tiltx_11, tiltx_19, n_adam=5000, n_lbfgs=80)

print(f"\n  Training time: {(datetime.now()-t0).total_seconds():.0f}s")

AR-PCNN: Autoregressive Prediction + Physics-Constrained Reconstruction
  Phase 1: 25 days
  Phase 2: 35 days
  Phase 3: 67 days
  Phase 4: 126 days

STEP 1: AR(1) Cross-Sensor Prediction
  Train: 213 days (measured) | Test: 40 days (predicted)
  11m: RMSE=0.0132 R2=0.7297 r=0.8556 (bias correction: +0.0028)
    AR_self=0.8038 AR_cross=-0.0624 T=0.00046 ph4xT=0.00088
  19m: RMSE=0.0129 R2=0.5206 r=0.7451 (bias correction: -0.0134)
    AR_self=-0.0537 AR_cross=0.6758 T=0.00468 ph4xT=-0.00152

STEP 2: CEI-PCNN (trained on all 253 days: measured + AR-predicted)
  Days 1-213: measured tiltx | Days 214-253: AR-predicted tiltx
  Pretraining EnvelopeNet...
    ep 0: shape=0.2441780 d4var=0.000000
    ep 3000: shape=0.0135441 d4var=0.000678
    ep 6000: shape=0.0001270 d4var=0.000014
    ep 9000: shape=0.0001906 d4var=0.000008
    Final shape_loss=0.000203
    env(0.0)=+0.00000 target=+0.00000
    env(7.5)=+0.48706 target=+0.48082
    env(15.0)=+0.00000 target=+0.00000
    env(22.5)=+0.95298 t

## Evaluate Accuracy

Compute PCNN predictions at all spatial positions and compare against FEM at sensor positions and 4 validation positions (x = 5, 10, 20, 25 m). Also compare discovered loads q(t) against FEM net UDL.

In [13]:
pcnn_sl, pcnn_df = solver.predict_at(gpos)
ip11 = np.argmin(np.abs(gpos-11)); ip19 = np.argmin(np.abs(gpos-19))
pred_11 = pcnn_sl[:,ip11]; pred_19 = pcnn_sl[:,ip19]
pq_AB = solver._q_AB.detach().numpy(); pq_BC = solver._q_BC.detach().numpy()

# ═══════════════════════════════════════════════════════════════
# RESULTS — SEPARATE TRAIN AND TEST
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"(A) ACCURACY — MEASURED PERIOD (Feb 22 – Sep 21, {N_TRAIN} days)")
print(f"{'='*70}")

# Sensor fit (training period — should be good since trained on these)
m_s_AB_tr = met(m11[:N_TRAIN], pred_11[:N_TRAIN])
m_s_BC_tr = met(m19[:N_TRAIN], pred_19[:N_TRAIN])
print(f"  Sensor: AB r={m_s_AB_tr['r']:.4f} R2={m_s_AB_tr['R2']:.4f} MAE={m_s_AB_tr['MAE']:.4f} | "
      f"BC r={m_s_BC_tr['r']:.4f} R2={m_s_BC_tr['R2']:.4f} MAE={m_s_BC_tr['MAE']:.4f}")

vs_tr = {}; vd_tr = {}
for xv in VAL_POS:
    ip = np.argmin(np.abs(gpos-xv))
    ms = met(fe_sl[:N_TRAIN,ip], pcnn_sl[:N_TRAIN,ip])
    md = met(fe_df[:N_TRAIN,ip], pcnn_df[:N_TRAIN,ip])
    vs_tr[xv] = ms; vd_tr[xv] = md
    print(f"  x={xv:.0f}m: sl r={ms['r']:.4f} R2={ms['R2']:.4f} MAE={ms['MAE']:.4f} Bias={ms['Bias']:+.3f} | "
          f"df r={md['r']:.4f} R2={md['R2']:.4f} MAE={md['MAE']:.4f} Bias={md['Bias']:+.3f}")

print(f"\n{'='*70}")
print(f"(B) ACCURACY — PREDICTED PERIOD (Sep 22 – Oct 31, {N_TEST} days)")
print(f"{'='*70}")

# AR prediction accuracy at sensors
print(f"  AR prediction at sensors:")
print(f"    11m: RMSE={ar_metrics['tiltx_11m']['rmse']:.4f} r={ar_metrics['tiltx_11m']['r']:.4f}")
print(f"    19m: RMSE={ar_metrics['tiltx_19m']['rmse']:.4f} r={ar_metrics['tiltx_19m']['r']:.4f}")

# PCNN reconstruction from AR predictions (test period)
print(f"  AR-PCNN reconstruction (test period):")
vs_te = {}; vd_te = {}
for xv in VAL_POS:
    ip = np.argmin(np.abs(gpos-xv))
    ms = met(fe_sl[N_TRAIN:,ip], pcnn_sl[N_TRAIN:,ip])
    md = met(fe_df[N_TRAIN:,ip], pcnn_df[N_TRAIN:,ip])
    vs_te[xv] = ms; vd_te[xv] = md
    print(f"    x={xv:.0f}m: sl r={ms['r']:.4f} R2={ms['R2']:.4f} MAE={ms['MAE']:.4f} Bias={ms['Bias']:+.3f} | "
          f"df r={md['r']:.4f} R2={md['R2']:.4f} MAE={md['MAE']:.4f} Bias={md['Bias']:+.3f}")

# Combined metrics (all N)
vs = {}; vd = {}
for xv in VAL_POS:
    ip = np.argmin(np.abs(gpos-xv))
    vs[xv] = met(fe_sl[:,ip], pcnn_sl[:,ip])
    vd[xv] = met(fe_df[:,ip], pcnn_df[:,ip])
m_s_AB = met(m11, pred_11); m_s_BC = met(m19, pred_19)
m_q_AB = met(fe_q_AB, pq_AB); m_q_BC = met(fe_q_BC, pq_BC)


(A) ACCURACY — MEASURED PERIOD (Feb 22 – Sep 21, 213 days)
  Sensor: AB r=0.9881 R2=0.9763 MAE=0.0295 | BC r=0.9799 R2=0.9597 MAE=0.0315
  x=5m: sl r=0.9868 R2=0.7793 MAE=0.0922 Bias=+0.092 | df r=0.9852 R2=0.8339 MAE=0.3817 Bias=-0.374
  x=10m: sl r=0.9861 R2=0.6650 MAE=0.1142 Bias=+0.114 | df r=0.9865 R2=0.6729 MAE=0.5697 Bias=+0.570
  x=20m: sl r=0.9717 R2=0.4675 MAE=0.1185 Bias=+0.118 | df r=0.9752 R2=0.3516 MAE=0.5536 Bias=-0.554
  x=25m: sl r=0.9607 R2=-2.2518 MAE=0.1398 Bias=+0.140 | df r=0.9702 R2=0.7405 MAE=0.4433 Bias=+0.414

(B) ACCURACY — PREDICTED PERIOD (Sep 22 – Oct 31, 40 days)
  AR prediction at sensors:
    11m: RMSE=0.0132 r=0.8556
    19m: RMSE=0.0129 r=0.7451
  AR-PCNN reconstruction (test period):
    x=5m: sl r=0.1181 R2=-18.6152 MAE=0.0865 Bias=+0.086 | df r=0.0436 R2=-6.2260 MAE=0.2325 Bias=-0.207
    x=10m: sl r=0.0904 R2=-12.8841 MAE=0.0685 Bias=+0.069 | df r=0.0509 R2=-17.9586 MAE=0.4234 Bias=+0.423
    x=20m: sl r=0.3418 R2=-15.9560 MAE=0.0661 Bias=+0.066 

## Evaluate Physics Compliance

Check boundary condition satisfaction (hard BCs should be ~1e-10), moment/shear continuity at the interior support, and PDE residual quality.

In [14]:
phys = evaluate_physics(solver, N)
print(f"\n{'='*70}\n(C) PHYSICS\n{'='*70}")
for k, v in phys['bc_violations_rms'].items():
    print(f"  {k:>10s}: {v:.2e}  [{'HARD' if v<1e-8 else 'SOFT'}]")
c = phys['continuity']
print(f"  Continuity: moment RMS={c['moment_rms']:.3e} | shear RMS={c['shear_rms']:.3e}")
for sn, p in phys['pde_residual'].items():
    print(f"  PDE {sn}: RMS={p['rms']:.3e} Max={p['max']:.3e}")

# ══════ PLOTS ══════
split_day = days[N_TRAIN]  # day where test period starts


(C) PHYSICS
        w(0): 1.64e-08  [SOFT]
       w(15): 1.38e-08  [SOFT]
       w(30): 7.26e-08  [SOFT]
       w'(0): 2.96e-09  [HARD]
      w'(15): 5.60e-07  [SOFT]
     w''(30): 1.08e-04  [SOFT]
  Continuity: moment RMS=1.288e-04 | shear RMS=3.781e-04
  PDE AB: RMS=1.762e-04 Max=2.439e-04
  PDE BC: RMS=8.639e-05 Max=9.475e-05


## Plot: Training Convergence

All loss components on log scale. L_data should decrease steadily; physics losses decrease as their weights ramp up.

In [15]:
# 1. Convergence
fig, ax = plt.subplots(figsize=(12,5))
ax.semilogy(hist['data'],'b-',lw=0.5,alpha=0.7,label='L_data')
ax.semilogy(hist['pde'],'r-',lw=0.5,alpha=0.7,label='L_pde')
ax.semilogy(hist['cont'],'g-',lw=0.5,alpha=0.7,label='L_continuity')
ax.semilogy(hist['mom'],'m-',lw=0.5,alpha=0.7,label="L_moment w''(30)")
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('AR-PCNN: CEI-PCNN Training Convergence (measured period only)',fontsize=12,fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'01_convergence.png'),dpi=200); plt.close()

# 2. AR Prediction + Sensor Fit
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('AR-PCNN: Measured (train) + Predicted (test) Tiltx',fontsize=13,fontweight='bold')
for ax,col,label,ar_p in [(axes[0],'tiltx_11m','AB x=11m',ar_pred['tiltx_11m']),
                           (axes[1],'tiltx_19m','BC x=19m',ar_pred['tiltx_19m'])]:
    # Measured period
    ax.plot(days[:N_TRAIN],m11[:N_TRAIN] if '11' in col else m19[:N_TRAIN],
            'b-',lw=1,alpha=0.7,label='Measured (train)')
    ax.plot(days[:N_TRAIN],pred_11[:N_TRAIN] if '11' in col else pred_19[:N_TRAIN],
            'c--',lw=0.8,alpha=0.5,label='PCNN recon (train)')
    # Test period
    actual_te = m11[N_TRAIN:] if '11' in col else m19[N_TRAIN:]
    pred_te = pred_11[N_TRAIN:] if '11' in col else pred_19[N_TRAIN:]
    ax.plot(days[N_TRAIN:],actual_te,'r-',lw=1.5,label='Measured (test, unseen)')
    ax.plot(days[N_TRAIN:],ar_p,'k--',lw=1.5,label=f'AR predicted')
    ax.plot(days[N_TRAIN:],pred_te,'g:',lw=1.5,label=f'PCNN from AR')
    ax.axvline(split_day,color='black',ls='-',lw=2,alpha=0.5)
    ax.axvspan(split_day,days[-1],alpha=0.05,color='red')
    ax.text(split_day+2,ax.get_ylim()[1]*0.95,'TEST',fontsize=10,fontweight='bold',color='red')
    _vlines(ax); ax.set_ylabel('tiltx (mm/m)'); ax.set_title(label)
    ax.legend(fontsize=7,ncol=3); ax.grid(True,alpha=0.3)
axes[1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'02_sensor_fit.png'),dpi=200); plt.close()

# 3. Validation time series (with train/test shading)
# 3. Validation time series (with train/test shading)
fig, axes = plt.subplots(4,2,figsize=(16,16))
fig.suptitle('Validation: AR-PCNN vs FEM (shaded = predicted period)',fontsize=13,fontweight='bold')
for row, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for col, fe, pc, ms_tr, ms_te, yl in [
        (0,fe_sl[:,ip],pcnn_sl[:,ip],vs_tr[xv],vs_te[xv],'Slope (mm/m)'),
        (1,fe_df[:,ip],pcnn_df[:,ip],vd_tr[xv],vd_te[xv],'Defl (mm)')]:
        axes[row,col].plot(days,fe,'b-',lw=1,alpha=0.7,label='FEM')
        axes[row,col].plot(days,pc,'r-',lw=1,alpha=0.7,
            label=f'AR-PCNN (train r={ms_tr["r"]:.3f}, test r={ms_te["r"]:.3f})')
        axes[row,col].axvline(split_day,color='k',ls='-',lw=2,alpha=0.3)
        axes[row,col].axvspan(split_day,days[-1],alpha=0.06,color='red')
        _vlines(axes[row,col]); axes[row,col].set_ylabel(yl)
        axes[row,col].set_title(f'x={xv:.0f}m'); axes[row,col].legend(fontsize=7); axes[row,col].grid(True,alpha=0.3)
axes[3,0].set_xlabel('Day'); axes[3,1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'03_validation_ts.png'),dpi=200); plt.close()

# 4. Validation scatter
fig, axes = plt.subplots(2,4,figsize=(18,8))
fig.suptitle('Scatter: PCNN vs FEM',fontsize=13,fontweight='bold')
for col, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for ph in [1,2,3,4]:
        mk = phases==ph
        axes[0,col].scatter(fe_sl[mk,ip],pcnn_sl[mk,ip],s=8,c=CP[ph],alpha=0.6,label=LP[ph] if col==0 else '')
        axes[1,col].scatter(fe_df[mk,ip],pcnn_df[mk,ip],s=8,c=CP[ph],alpha=0.6)
    for r in [0,1]:
        fe = fe_sl[:,ip] if r==0 else fe_df[:,ip]
        lm=[fe.min(),fe.max()]; axes[r,col].plot(lm,lm,'k--'); axes[r,col].grid(True,alpha=0.3)
    axes[0,col].set_title(f'x={xv:.0f}m Slope',fontsize=9)
    axes[1,col].set_title(f'x={xv:.0f}m Defl',fontsize=9); axes[1,col].set_xlabel('FEM')
axes[0,0].legend(fontsize=5,ncol=2); axes[0,0].set_ylabel('PCNN Slope'); axes[1,0].set_ylabel('PCNN Defl')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'04_validation_scatter.png'),dpi=200); plt.close()

# 5. Mean profiles
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Mean Profiles: FEM vs AR-PCNN',fontsize=13,fontweight='bold')
for ax,fe,pc,yl in [(axes[0],fe_sl,pcnn_sl,'Slope (mm/m)'),(axes[1],fe_df,pcnn_df,'Defl (mm)')]:
    ax.plot(gpos,fe.mean(0),'k-',lw=2,label='FEM'); ax.plot(gpos,pc.mean(0),'r--',lw=2,label='AR-PCNN')
    ax.axvline(11,color='blue',ls=':',lw=1.5,alpha=0.7); ax.axvline(19,color='blue',ls=':',lw=1.5,alpha=0.7)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.set_ylabel(yl); ax.legend(); ax.grid(True,alpha=0.3)
axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'05_mean_profiles.png'),dpi=200); plt.close()

# 6. Load comparison
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Load: PCNN q(t) vs FEM',fontsize=13,fontweight='bold')
for col,sn,fq,pq,mq in [(0,'AB',fe_q_AB,pq_AB,m_q_AB),(1,'BC',fe_q_BC,pq_BC,m_q_BC)]:
    axes[0,col].plot(days,fq,'b-',lw=1,alpha=0.7,label='FEM')
    axes[0,col].plot(days,pq,'r-',lw=1,alpha=0.7,label=f'PCNN (r={mq["r"]:.3f})')
    for d in [46,81,148]: axes[0,col].axvline(d,color='gray',ls='--',lw=0.5)
    axes[0,col].set_ylabel('q (kN/m)'); axes[0,col].set_title(f'{sn} Load'); axes[0,col].legend(fontsize=7); axes[0,col].grid(True,alpha=0.3)
    for ph in [1,2,3,4]:
        mk=phases==ph; axes[1,col].scatter(fq[mk],pq[mk],s=12,c=CP[ph],alpha=0.6,label=LP[ph])
    lm=[min(fq.min(),pq.min()),max(fq.max(),pq.max())]; axes[1,col].plot(lm,lm,'k--')
    axes[1,col].set_xlabel('FEM (kN/m)'); axes[1,col].set_ylabel('PCNN (kN/m)')
    axes[1,col].set_title(f'{sn} RMSE={mq["RMSE"]:.2f}'); axes[1,col].legend(fontsize=6); axes[1,col].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'06_load.png'),dpi=200); plt.close()

# 7. 3D slope + defl
Xg,Tg = np.meshgrid(gpos,days)
for data,zl,fn in [(pcnn_sl,'Slope','07_3d_slope.png'),(pcnn_df,'Defl','08_3d_defl.png')]:
    fig=plt.figure(figsize=(16,10)); ax3=fig.add_subplot(111,projection='3d')
    ax3.plot_surface(Xg,Tg,data,cmap='RdBu_r',alpha=0.85,rstride=2,cstride=1,linewidth=0)
    ax3.set_xlabel('x (m)'); ax3.set_ylabel('Day'); ax3.set_zlabel(zl); ax3.view_init(25,-60)
    plt.tight_layout(); plt.savefig(os.path.join(OUT,fn),dpi=200); plt.close()

# 9. Accuracy bar (r, RMSE, Bias)
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Accuracy Metrics',fontsize=13,fontweight='bold')
xp = np.arange(len(VAL_POS)); wd = 0.35
for ax,metric,ylabel in [(axes[0],'r','Pearson r'),(axes[1],'RMSE','RMSE'),(axes[2],'Bias','Bias')]:
    sv=[vs[xv][metric] for xv in VAL_POS]; dv=[vd[xv][metric] for xv in VAL_POS]
    b1=ax.bar(xp-wd/2,sv,wd,label='Slope',color='steelblue',alpha=0.8)
    b2=ax.bar(xp+wd/2,dv,wd,label='Deflection',color='coral',alpha=0.8)
    for b,vals in [(b1,sv),(b2,dv)]:
        for bar,v in zip(b,vals):
            ax.annotate(f'{v:.3f}',xy=(bar.get_x()+bar.get_width()/2,bar.get_height()),
                        ha='center',va='bottom' if v>=0 else 'top',fontsize=7)
    ax.set_xticks(xp); ax.set_xticklabels([f'{xv:.0f}m' for xv in VAL_POS])
    ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
    if metric=='Bias': ax.axhline(0,color='k',lw=0.5)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'09_accuracy_bar.png'),dpi=200); plt.close()

# 10. Physics bar
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Physical Consistency',fontsize=13,fontweight='bold')
bc_n=list(phys['bc_violations_rms'].keys()); bc_v=[phys['bc_violations_rms'][k] for k in bc_n]
colors=['green' if v<1e-8 else 'orange' if v<0.01 else 'red' for v in bc_v]
axes[0].barh(bc_n,bc_v,color=colors,alpha=0.8); axes[0].set_xscale('log')
axes[0].set_xlabel('RMS'); axes[0].set_title('BC Compliance'); axes[0].grid(True,alpha=0.3)
cn=['Moment','Shear']; cr=[c['moment_rms'],c['shear_rms']]; cm=[c['moment_max'],c['shear_max']]
xp2=np.arange(2)
axes[1].bar(xp2-0.17,cr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[1].bar(xp2+0.17,cm,0.34,label='Max',color='coral',alpha=0.8)
axes[1].set_xticks(xp2); axes[1].set_xticklabels(cn); axes[1].set_title('Continuity x=15'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
pn=list(phys['pde_residual'].keys()); pr=[phys['pde_residual'][k]['rms'] for k in pn]; pm=[phys['pde_residual'][k]['max'] for k in pn]
xp3=np.arange(len(pn))
axes[2].bar(xp3-0.17,pr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[2].bar(xp3+0.17,pm,0.34,label='Max',color='coral',alpha=0.8)
axes[2].set_xticks(xp3); axes[2].set_xticklabels([f'Span {k}' for k in pn]); axes[2].set_title('PDE Residual'); axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'10_physics_bar.png'),dpi=200); plt.close()

# 11. Daily profiles: Feb 26, Apr 26, Jul 26, Oct 26
target_days = [25, 85, 176, 268]  # day numbers from Feb 1
sel = [np.argmin(np.abs(days-td)) for td in target_days]
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Daily Profiles: FEM (solid) vs AR-PCNN (dashed)',fontsize=13,fontweight='bold')
for j,di in enumerate(sel):
    ds = day_to_date(days[di]); cc = DC[j]
    axes[0].plot(gpos,fe_sl[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[0].plot(gpos,pcnn_sl[di],'--',color=cc,lw=2)
    axes[1].plot(gpos,fe_df[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[1].plot(gpos,pcnn_df[di],'--',color=cc,lw=2)
axes[0].plot([],[],'-k',lw=2,label='FEM'); axes[0].plot([],[],'--k',lw=2,label='AR-PCNN')
for ax in axes:
    ax.axvline(11,color='blue',ls=':',lw=1,alpha=0.5); ax.axvline(19,color='red',ls=':',lw=1,alpha=0.5)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.legend(fontsize=8,ncol=3); ax.grid(True,alpha=0.3)
axes[0].set_ylabel('Slope (mm/m)'); axes[1].set_ylabel('Deflection (mm)'); axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'11_daily_profiles.png'),dpi=200); plt.close()

# Save
df_out = pd.DataFrame({'day':days,'phase':phases,'tiltx_11m':m11,'tiltx_19m':m19,
    'ar_pred_11m':tiltx_11,'ar_pred_19m':tiltx_19,
    'is_predicted':[0]*N_TRAIN+[1]*N_TEST,
    'pcnn_pred_11m':pred_11,'pcnn_pred_19m':pred_19,'q_AB':pq_AB,'q_BC':pq_BC,
    'fe_q_AB':fe_q_AB,'fe_q_BC':fe_q_BC})
for ip,xv in enumerate(gpos):
    df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
df_out.to_csv(os.path.join(OUT,'ar_pcnn_results.csv'),index=False)

with open(os.path.join(OUT,'ar_pcnn_summary.json'),'w') as f:
    json.dump({'method':'AR-PCNN (AR(1) cross-sensor + CEI-PCNN)',
        'N_TRAIN':N_TRAIN,'N_TEST':N_TEST,
        'ar_model':{'type':'Ridge(alpha=0.1)','features':9,
            '11m':{'rmse':ar_metrics['tiltx_11m']['rmse'],'r':ar_metrics['tiltx_11m']['r']},
            '19m':{'rmse':ar_metrics['tiltx_19m']['rmse'],'r':ar_metrics['tiltx_19m']['r']}},
        'pcnn_train':{'sensor':{'AB':m_s_AB_tr,'BC':m_s_BC_tr},
            'validation':{f'x{xv:.0f}m':{'slope':vs_tr[xv],'defl':vd_tr[xv]} for xv in VAL_POS}},
        'pcnn_test':{'validation':{f'x{xv:.0f}m':{'slope':vs_te[xv],'defl':vd_te[xv]} for xv in VAL_POS}},
        'physics':phys},f,indent=2,default=str)

print(f"\n  11 plots + CSV + JSON saved to {OUT}/")
print(f"  AR-PCNN pipeline complete.")
print(f"  Runtime: {(datetime.now()-t0).total_seconds():.0f}s")

findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.



  11 plots + CSV + JSON saved to C:\Users\ridoc\OneDrive\Desktop\Folders\SMACCs\Thesis\Physics Guided Framework\Draft\Models\outputs\processing\Tiltx\AR_PCNN/
  AR-PCNN pipeline complete.
  Runtime: 2588s


C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\4220501126.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\4220501126.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\4220501126.py:165: PerformanceWarning: DataFrame is highly fragmented.

## Plot: Validation Time Series

PCNN vs FEM at 4 uninstrumented validation positions. Left: slope, Right: deflection. These positions are NOT used during training.

In [16]:
# 3. Validation time series (with train/test shading)
# 3. Validation time series (with train/test shading)
fig, axes = plt.subplots(4,2,figsize=(16,16))
fig.suptitle('Validation: AR-PCNN vs FEM (shaded = predicted period)',fontsize=13,fontweight='bold')
for row, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for col, fe, pc, ms_tr, ms_te, yl in [
        (0,fe_sl[:,ip],pcnn_sl[:,ip],vs_tr[xv],vs_te[xv],'Slope (mm/m)'),
        (1,fe_df[:,ip],pcnn_df[:,ip],vd_tr[xv],vd_te[xv],'Defl (mm)')]:
        axes[row,col].plot(days,fe,'b-',lw=1,alpha=0.7,label='FEM')
        axes[row,col].plot(days,pc,'r-',lw=1,alpha=0.7,
            label=f'AR-PCNN (train r={ms_tr["r"]:.3f}, test r={ms_te["r"]:.3f})')
        axes[row,col].axvline(split_day,color='k',ls='-',lw=2,alpha=0.3)
        axes[row,col].axvspan(split_day,days[-1],alpha=0.06,color='red')
        _vlines(axes[row,col]); axes[row,col].set_ylabel(yl)
        axes[row,col].set_title(f'x={xv:.0f}m'); axes[row,col].legend(fontsize=7); axes[row,col].grid(True,alpha=0.3)
axes[3,0].set_xlabel('Day'); axes[3,1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'03_validation_ts.png'),dpi=200); plt.close()

## Plot: Validation Scatter (Phase-Coloured)

Scatter plots of PCNN vs FEM, coloured by loading phase. Points on the diagonal = perfect agreement.

In [17]:
# 4. Validation scatter
fig, axes = plt.subplots(2,4,figsize=(18,8))
fig.suptitle('Scatter: PCNN vs FEM',fontsize=13,fontweight='bold')
for col, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for ph in [1,2,3,4]:
        mk = phases==ph
        axes[0,col].scatter(fe_sl[mk,ip],pcnn_sl[mk,ip],s=8,c=CP[ph],alpha=0.6,label=LP[ph] if col==0 else '')
        axes[1,col].scatter(fe_df[mk,ip],pcnn_df[mk,ip],s=8,c=CP[ph],alpha=0.6)
    for r in [0,1]:
        fe = fe_sl[:,ip] if r==0 else fe_df[:,ip]
        lm=[fe.min(),fe.max()]; axes[r,col].plot(lm,lm,'k--'); axes[r,col].grid(True,alpha=0.3)
    axes[0,col].set_title(f'x={xv:.0f}m Slope',fontsize=9)
    axes[1,col].set_title(f'x={xv:.0f}m Defl',fontsize=9); axes[1,col].set_xlabel('FEM')
axes[0,0].legend(fontsize=5,ncol=2); axes[0,0].set_ylabel('PCNN Slope'); axes[1,0].set_ylabel('PCNN Defl')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'04_validation_scatter.png'),dpi=200); plt.close()

## Plot: Mean Spatial Profiles

Mean slope and deflection profiles (averaged over all 253 days). Reveals systematic spatial bias from trial function shape mismatch.

In [18]:
# 5. Mean profiles
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Mean Profiles: FEM vs AR-PCNN',fontsize=13,fontweight='bold')
for ax,fe,pc,yl in [(axes[0],fe_sl,pcnn_sl,'Slope (mm/m)'),(axes[1],fe_df,pcnn_df,'Defl (mm)')]:
    ax.plot(gpos,fe.mean(0),'k-',lw=2,label='FEM'); ax.plot(gpos,pc.mean(0),'r--',lw=2,label='AR-PCNN')
    ax.axvline(11,color='blue',ls=':',lw=1.5,alpha=0.7); ax.axvline(19,color='blue',ls=':',lw=1.5,alpha=0.7)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.set_ylabel(yl); ax.legend(); ax.grid(True,alpha=0.3)
axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'05_mean_profiles.png'),dpi=200); plt.close()

## Plot: Load Discovery

PCNN-discovered load q(t) vs FEM net UDL. Good load discovery (high r) confirms the PDE loss is working correctly.

In [19]:
# 6. Load comparison
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Load: PCNN q(t) vs FEM',fontsize=13,fontweight='bold')
for col,sn,fq,pq,mq in [(0,'AB',fe_q_AB,pq_AB,m_q_AB),(1,'BC',fe_q_BC,pq_BC,m_q_BC)]:
    axes[0,col].plot(days,fq,'b-',lw=1,alpha=0.7,label='FEM')
    axes[0,col].plot(days,pq,'r-',lw=1,alpha=0.7,label=f'PCNN (r={mq["r"]:.3f})')
    for d in [46,81,148]: axes[0,col].axvline(d,color='gray',ls='--',lw=0.5)
    axes[0,col].set_ylabel('q (kN/m)'); axes[0,col].set_title(f'{sn} Load'); axes[0,col].legend(fontsize=7); axes[0,col].grid(True,alpha=0.3)
    for ph in [1,2,3,4]:
        mk=phases==ph; axes[1,col].scatter(fq[mk],pq[mk],s=12,c=CP[ph],alpha=0.6,label=LP[ph])
    lm=[min(fq.min(),pq.min()),max(fq.max(),pq.max())]; axes[1,col].plot(lm,lm,'k--')
    axes[1,col].set_xlabel('FEM (kN/m)'); axes[1,col].set_ylabel('PCNN (kN/m)')
    axes[1,col].set_title(f'{sn} RMSE={mq["RMSE"]:.2f}'); axes[1,col].legend(fontsize=6); axes[1,col].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'06_load.png'),dpi=200); plt.close()

## Plot: 3D Spatio-Temporal Surfaces

3D surface plots of slope(x,t) and deflection(x,t) showing the full reconstructed field over all days and positions.

In [20]:
# 7. 3D slope + defl
Xg,Tg = np.meshgrid(gpos,days)
for data,zl,fn in [(pcnn_sl,'Slope','07_3d_slope.png'),(pcnn_df,'Defl','08_3d_defl.png')]:
    fig=plt.figure(figsize=(16,10)); ax3=fig.add_subplot(111,projection='3d')
    ax3.plot_surface(Xg,Tg,data,cmap='RdBu_r',alpha=0.85,rstride=2,cstride=1,linewidth=0)
    ax3.set_xlabel('x (m)'); ax3.set_ylabel('Day'); ax3.set_zlabel(zl); ax3.view_init(25,-60)
    plt.tight_layout(); plt.savefig(os.path.join(OUT,fn),dpi=200); plt.close()

## Plot: Accuracy Metrics Bar Chart

Bar chart of Pearson r, RMSE, and Bias at each validation position, split by slope and deflection.

In [21]:
# 9. Accuracy bar (r, RMSE, Bias)
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Accuracy Metrics',fontsize=13,fontweight='bold')
xp = np.arange(len(VAL_POS)); wd = 0.35
for ax,metric,ylabel in [(axes[0],'r','Pearson r'),(axes[1],'RMSE','RMSE'),(axes[2],'Bias','Bias')]:
    sv=[vs[xv][metric] for xv in VAL_POS]; dv=[vd[xv][metric] for xv in VAL_POS]
    b1=ax.bar(xp-wd/2,sv,wd,label='Slope',color='steelblue',alpha=0.8)
    b2=ax.bar(xp+wd/2,dv,wd,label='Deflection',color='coral',alpha=0.8)
    for b,vals in [(b1,sv),(b2,dv)]:
        for bar,v in zip(b,vals):
            ax.annotate(f'{v:.3f}',xy=(bar.get_x()+bar.get_width()/2,bar.get_height()),
                        ha='center',va='bottom' if v>=0 else 'top',fontsize=7)
    ax.set_xticks(xp); ax.set_xticklabels([f'{xv:.0f}m' for xv in VAL_POS])
    ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
    if metric=='Bias': ax.axhline(0,color='k',lw=0.5)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'09_accuracy_bar.png'),dpi=200); plt.close()

## Plot: Physics Compliance

Three-panel chart: (1) BC violations on log scale, (2) continuity jumps (RMS/Max), (3) PDE residual per span.

In [22]:
# 10. Physics bar
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Physical Consistency',fontsize=13,fontweight='bold')
bc_n=list(phys['bc_violations_rms'].keys()); bc_v=[phys['bc_violations_rms'][k] for k in bc_n]
colors=['green' if v<1e-8 else 'orange' if v<0.01 else 'red' for v in bc_v]
axes[0].barh(bc_n,bc_v,color=colors,alpha=0.8); axes[0].set_xscale('log')
axes[0].set_xlabel('RMS'); axes[0].set_title('BC Compliance'); axes[0].grid(True,alpha=0.3)
cn=['Moment','Shear']; cr=[c['moment_rms'],c['shear_rms']]; cm=[c['moment_max'],c['shear_max']]
xp2=np.arange(2)
axes[1].bar(xp2-0.17,cr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[1].bar(xp2+0.17,cm,0.34,label='Max',color='coral',alpha=0.8)
axes[1].set_xticks(xp2); axes[1].set_xticklabels(cn); axes[1].set_title('Continuity x=15'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
pn=list(phys['pde_residual'].keys()); pr=[phys['pde_residual'][k]['rms'] for k in pn]; pm=[phys['pde_residual'][k]['max'] for k in pn]
xp3=np.arange(len(pn))
axes[2].bar(xp3-0.17,pr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[2].bar(xp3+0.17,pm,0.34,label='Max',color='coral',alpha=0.8)
axes[2].set_xticks(xp3); axes[2].set_xticklabels([f'Span {k}' for k in pn]); axes[2].set_title('PDE Residual'); axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'10_physics_bar.png'),dpi=200); plt.close()

## Plot: Daily Profiles

Selected daily profiles from different phases: FEM (solid) vs PCNN (dashed). Shows day-to-day variation, not just mean shape.

In [23]:
# 11. Daily profiles: Feb 26, Apr 26, Jul 26, Oct 26
target_days = [25, 85, 176, 268]  # day numbers from Feb 1
sel = [np.argmin(np.abs(days-td)) for td in target_days]
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Daily Profiles: FEM (solid) vs AR-PCNN (dashed)',fontsize=13,fontweight='bold')
for j,di in enumerate(sel):
    ds = day_to_date(days[di]); cc = DC[j]
    axes[0].plot(gpos,fe_sl[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[0].plot(gpos,pcnn_sl[di],'--',color=cc,lw=2)
    axes[1].plot(gpos,fe_df[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[1].plot(gpos,pcnn_df[di],'--',color=cc,lw=2)
axes[0].plot([],[],'-k',lw=2,label='FEM'); axes[0].plot([],[],'--k',lw=2,label='AR-PCNN')
for ax in axes:
    ax.axvline(11,color='blue',ls=':',lw=1,alpha=0.5); ax.axvline(19,color='red',ls=':',lw=1,alpha=0.5)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.legend(fontsize=8,ncol=3); ax.grid(True,alpha=0.3)
axes[0].set_ylabel('Slope (mm/m)'); axes[1].set_ylabel('Deflection (mm)'); axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'11_daily_profiles.png'),dpi=200); plt.close()

## Save Results

Save pcnn_results.csv (all predictions at all positions) and pcnn_summary.json (accuracy + physics metrics).

In [24]:
# Save
df_out = pd.DataFrame({'day':days,'phase':phases,'tiltx_11m':m11,'tiltx_19m':m19,
    'ar_pred_11m':tiltx_11,'ar_pred_19m':tiltx_19,
    'is_predicted':[0]*N_TRAIN+[1]*N_TEST,
    'pcnn_pred_11m':pred_11,'pcnn_pred_19m':pred_19,'q_AB':pq_AB,'q_BC':pq_BC,
    'fe_q_AB':fe_q_AB,'fe_q_BC':fe_q_BC})
for ip,xv in enumerate(gpos):
    df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
df_out.to_csv(os.path.join(OUT,'ar_pcnn_results.csv'),index=False)

with open(os.path.join(OUT,'ar_pcnn_summary.json'),'w') as f:
    json.dump({'method':'AR-PCNN (AR(1) cross-sensor + CEI-PCNN)',
        'N_TRAIN':N_TRAIN,'N_TEST':N_TEST,
        'ar_model':{'type':'Ridge(alpha=0.1)','features':9,
            '11m':{'rmse':ar_metrics['tiltx_11m']['rmse'],'r':ar_metrics['tiltx_11m']['r']},
            '19m':{'rmse':ar_metrics['tiltx_19m']['rmse'],'r':ar_metrics['tiltx_19m']['r']}},
        'pcnn_train':{'sensor':{'AB':m_s_AB_tr,'BC':m_s_BC_tr},
            'validation':{f'x{xv:.0f}m':{'slope':vs_tr[xv],'defl':vd_tr[xv]} for xv in VAL_POS}},
        'pcnn_test':{'validation':{f'x{xv:.0f}m':{'slope':vs_te[xv],'defl':vd_te[xv]} for xv in VAL_POS}},
        'physics':phys},f,indent=2,default=str)

print(f"\n  11 plots + CSV + JSON saved to {OUT}/")
print(f"  AR-PCNN pipeline complete.")
print(f"  Runtime: {(datetime.now()-t0).total_seconds():.0f}s")


  11 plots + CSV + JSON saved to C:\Users\ridoc\OneDrive\Desktop\Folders\SMACCs\Thesis\Physics Guided Framework\Draft\Models\outputs\processing\Tiltx\AR_PCNN/
  AR-PCNN pipeline complete.
  Runtime: 2660s


C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\3657510723.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\3657510723.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_19976\3657510723.py:8: PerformanceWarning: DataFrame is highly fragmented.  This